<a id="optional-fno-ablation"></a>
# 선택 실습 — FNO의 푸리에 모드 수에 따른 계산 비용과 오차 비교

이 노트북은 [02_Poisson_FNO.ipynb](../02_Poisson_FNO.ipynb)를 마친 뒤 강사 안내에 따라 진행합니다. 필수 FNO 학습과 동시에 실행하지 않습니다.

**실험 질문:** 동일한 데이터와 학습 조건에서 모델의 `fno_modes`를 6에서 12로 늘리면 전체 실행 시간, 파라미터 수, 최대 PyTorch 메모리 사용량, 테스트 오차가 어떻게 달라질까요?

**사전 가설:** 데이터가 `max_mode=6`으로 생성되어 `fno_modes=6`만으로 주요 주파수 성분을 표현할 수 있다면, 모드를 12개로 늘렸을 때 오차 감소보다 계산 비용 증가가 더 클 수 있습니다. 측정 결과가 이 가설과 일치하는지 기록합니다.


## 1. 환경과 파일 확인

`labs/poisson_fno`를 찾고, 두 비교용 설정 파일에서 `fno_modes`와 출력 이름 외의 조건이 같은지 확인합니다.


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import subprocess
import sys
import time

launch_dir = Path.cwd().resolve()
REPO_ROOT = next(
    (
        candidate
        for candidate in (launch_dir, *launch_dir.parents)
        if (candidate / "labs" / "poisson_fno" / "train_fno.py").is_file()
    ),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError(
        "labs/poisson_fno/train_fno.py를 찾지 못했습니다. 과정 root 또는 optional 폴더에서 notebook을 여세요."
    )

LAB_DIR = REPO_ROOT / "labs" / "poisson_fno"
if str(LAB_DIR) not in sys.path:
    sys.path.insert(0, str(LAB_DIR))

import torch
import physicsnemo
import physicsnemo.sym
from notebook_utils import assert_ablation_configs_match, ensure_profile_dataset

config_6 = LAB_DIR / "conf" / "config_FNO_ablation_6.yaml"
config_12 = LAB_DIR / "conf" / "config_FNO_ablation_12.yaml"
differences = assert_ablation_configs_match(config_6, config_12)

print(f"과정 루트      : {REPO_ROOT}")
print(f"FNO 실습 폴더  : {LAB_DIR}")
print(f"GPU            : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CUDA 사용 불가'}")
print("두 설정에서 제어한 차이:")
for key, values in differences.items():
    print(f"  {key}: {values[0]!r} → {values[1]!r}")


## 2. 비교에 사용할 공통 축소 데이터셋 준비

두 실험은 같은 `64×64` 학습·검증·테스트 데이터를 재사용합니다. 데이터 생성기의 `max_mode=6`은 정답 데이터를 만들 때 포함할 최대 주파수 모드를 뜻하며, 비교할 모델의 `fno_modes=6/12`와는 서로 다른 설정입니다.


In [ ]:
PROFILE_INFO = ensure_profile_dataset(
    LAB_DIR,
    "recovery",
    force=False,
    device="cuda" if torch.cuda.is_available() else "auto",
)
print(f"공유 데이터셋: {PROFILE_INFO['dataset_dir']}")


## 3. 푸리에 모드 6개와 12개를 차례로 실행

축소 데이터셋, 난수 시드 `2026`, 채널 너비 `32`, FNO 층 `4`, 배치 크기 `32`, 최적화 방법, 학습률 조정 방식, `200단계` 학습은 두 실험에서 동일하게 유지합니다. 각 실행 결과는 실행 시각을 이름에 넣은 별도 출력 폴더에 저장합니다.

전체 실행 시간에는 CUDA 초기화, 파일 입출력, 검증 시간이 포함될 수 있습니다. 실행 순서의 영향을 통제하지 않았으므로 작은 차이에 일반적인 의미를 부여하지 않습니다.


In [ ]:
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
experiments = [
    {"modes": 6, "config": "config_FNO_ablation_6"},
    {"modes": 12, "config": "config_FNO_ablation_12"},
]
RESULTS = []

for experiment in experiments:
    modes = experiment["modes"]
    output_dir = LAB_DIR / "outputs" / "ksc_fno_ablation" / RUN_ID / f"modes_{modes}"
    metrics_path = output_dir / "final_state_test_metrics.json"
    command = [
        sys.executable,
        "train_fno.py",
        "--config-name",
        experiment["config"],
        f"network_dir={output_dir}",
        f"custom.metrics_file={metrics_path}",
    ]

    print("\n" + "=" * 72)
    print(f"FNO modes={modes} 실행")
    print("명령:", " ".join(str(part) for part in command))
    started = time.perf_counter()
    completed = subprocess.run(command, cwd=LAB_DIR)
    wall_seconds = time.perf_counter() - started
    if completed.returncode != 0:
        raise RuntimeError(f"modes={modes} 학습 실패 (exit code={completed.returncode})")

    payload = json.loads(metrics_path.read_text(encoding="utf-8"))
    assert payload["profile"] == "recovery"
    assert payload["random_seed"] == 2026
    assert payload["model"]["fno_modes"] == modes
    RESULTS.append(
        {
            "modes": modes,
            "parameters": payload["model"]["trainable_parameters"],
            "wall_seconds": wall_seconds,
            "peak_memory_bytes": payload["runtime_observation"]["peak_memory_allocated_bytes"],
            "relative_l2": payload["metrics"]["relative_l2"],
            "rmse": payload["metrics"]["rmse"],
            "metrics_path": metrics_path,
        }
    )

print("\n두 실험이 완료되었습니다.")


## 4. 결과 비교

상대 L2 오차가 작을수록 이 테스트 데이터에서 예측이 정답에 더 가깝다는 뜻입니다. 파라미터 수, 최대 메모리 사용량, 전체 실행 시간은 계산 비용을 서로 다른 관점에서 보여 줍니다. 여기서 최대 메모리는 PyTorch 프로세스가 할당했다고 보고한 값이며 GPU 전체 사용량은 아닙니다.


In [ ]:
import matplotlib.pyplot as plt

header = f"{'Modes':>7} {'Parameters':>14} {'Wall min':>12} {'Peak GiB':>11} {'Relative L2':>16} {'RMSE':>14}"
print(header)
for result in RESULTS:
    peak_gib = (
        result["peak_memory_bytes"] / 2**30
        if result["peak_memory_bytes"] is not None
        else float("nan")
    )
    print(
        f"{result['modes']:7d} {result['parameters']:14,d} "
        f"{result['wall_seconds'] / 60:12.2f} {peak_gib:11.2f} "
        f"{result['relative_l2']:16.6e} {result['rmse']:14.6e}"
    )

labels = [f"modes={result['modes']}" for result in RESULTS]
figure, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
axes[0].bar(labels, [r["wall_seconds"] / 60 for r in RESULTS], color=["#76B900", "#1f6feb"])
axes[0].set(title="Wall time", ylabel="min")
axes[1].bar(labels, [r["parameters"] for r in RESULTS], color=["#76B900", "#1f6feb"])
axes[1].set(title="Trainable parameters", ylabel="count")
axes[2].bar(labels, [r["relative_l2"] for r in RESULTS], color=["#76B900", "#1f6feb"])
axes[2].set(title="Test error", ylabel="relative L2")
plt.show()


## 5. 결과 해석과 반복 실험

1. 푸리에 모드 12개를 사용한 모델은 6개를 사용한 모델보다 파라미터 수, 메모리 사용량, 전체 실행 시간이 얼마나 늘었나요?
2. 상대 L2 오차의 차이는 계산 비용 증가에 비해 컸나요, 작았나요?
3. 데이터의 `max_mode=6`에서 두 모델의 차이가 작다면, 모드 6개만으로 이 데이터의 주파수 정보를 충분히 표현했다는 가설과 맞나요?
4. 첫 실행에 포함되는 CUDA 초기화 비용을 통제하려면 실행 순서 무작위화나 준비 실행(warm-up)을 어떻게 적용할 수 있을까요?
5. 여러 난수 시드와 반복 실행에서 평균·분산을 계산하면 결론의 신뢰도가 어떻게 달라질까요?

결과는 난수 시드 하나, 200단계 학습, 지정된 데이터셋과 현재 하드웨어 상태에 대한 값으로 기록합니다. 반복 실험을 하지 않은 결과는 해당 실행의 관찰값으로 해석합니다. [PhysicsNeMo 모듈 지도](../README.md)로 돌아가 전체 과정을 정리합니다.


---

## 저작자 표시와 라이선스

이 KSC 실습은 OpenHackathons AI-Powered-Physics-Bootcamp 자료를 바탕으로 개작했습니다. 각 파일에 적힌 기존 저작권과 라이선스 고지는 그대로 적용됩니다.
